# A-Maze-ing

## Read the configuration file

Just open the file, split the options turn them into a dictionary.

In [12]:
with open("config.txt") as f:
    txt = f.read()
    opts = txt.split("\n")
    tup = [tuple(i.split("=")) for i in opts]
    conf = {i[0]: i[1] for i in tup}
print(conf)
algo = conf.get("ALGORITHM")
print(algo.lower())

{'HEIGHT': '20', 'WIDTH': '20', 'ENTRY': '0,0', 'EXIT': '19,19', 'OUTPUT_FILE': 'maze.txt', 'PERFECT': 'True', 'ALGORITHM': 'Wilson'}
wilson


## Create a $h \times w$ array

Two different arrays:

* The maze itself.
* The visited cells.

The 42 was hardcoded.

In [2]:
from numpy import zeros, ones, mean, array


size = (int(conf["HEIGHT"]), int(conf["WIDTH"]))
maze = ones(size, int) * 15
visited = zeros(size, int)
perfect = bool(conf["PERFECT"])
if visited.shape[0] < 6 or visited.shape[1] < 8:
    is_42 = 0
    print("The '42' cannot be printed in a maze this size.")
else:
    is_42 = 1
    hs, ws = (visited.shape[0] - 1) // 2 - 2, (visited.shape[1] - 1) // 2 - 3
    pos_42 = (
        (hs, ws), (hs + 1, ws), (hs + 2, ws), (hs + 2, ws + 1),
        (hs + 2, ws + 2), (hs + 3, ws + 2), (hs + 4, ws + 2),
        (hs, ws + 4), (hs, ws + 5), (hs, ws + 6), (hs + 1, ws + 6),
        (hs + 2, ws + 6), (hs + 2, ws + 5), (hs + 2, ws + 4),
        (hs + 3, ws + 4), (hs + 4, ws + 4), (hs + 4, ws + 5),
        (hs + 4, ws + 6)
    )
    for i in pos_42:
        visited[i[0]][i[1]] = -1
print(pos_42)
print(visited)
for i in maze:
    print(i)

((7, 6), (8, 6), (9, 6), (9, 7), (9, 8), (10, 8), (11, 8), (7, 10), (7, 11), (7, 12), (8, 12), (9, 12), (9, 11), (9, 10), (10, 10), (11, 10), (11, 11), (11, 12))
[[ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0 -1  0  0  0 -1 -1 -1  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0 -1  0  0  0  0  0 -1  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0 -1 -1 -1  0 -1 -1 -1  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0 -1  0 -1  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0 -1  0 -1 -1 -1  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0

## Manage errors

Checks:

* Maze dimensions must be positive.
* Maze entrance and exit are not in the same tile.
* Maze entrance and exit are inside the maze.
* Maze entrance and exit are not inside forbidden tiles.
* All maze walls in a tile are coherent with the neighbouring tiles.

In [3]:
from numpy.typing import NDArray


class MazeError(Exception):
    def __init__(self, msg: str):
        self.msg = msg


start = conf["ENTRY"].split(",")
end = conf["EXIT"].split(",")
entr = (int(start[0]), int(start[1]))
exit = (int(end[0]), int(end[1]))

if entr == exit:
    raise MazeError("Maze entrance and exit must be different")

if size[0] <= 0 or size[1] <= 0:
    raise MazeError("Maze dimensions must be positive")

in_bounds = [
    entr[0] >= 0,
    entr[1] >= 0,
    entr[0] < size[0],
    entr[1] < size[1],
    exit[0] >= 0,
    exit[1] >= 0,
    exit[0] < size[0],
    exit[1] < size[1]
]
if False in in_bounds:
    raise MazeError("Entrance or exit of the maze seem to be out of bounds")

if visited.shape[0] > 5 or visited.shape[1] > 7:
    if True in [entr in pos_42, exit in pos_42]:
        raise MazeError("Entrance or exit of the maze seem to be inside the 42")


def get_walls(n: int) -> tuple[int]:
    if n > 15 or n < 0:
        raise ValueError
    bits = []
    bits.append(n // 8)
    n %= 8
    bits.append(n // 4)
    n %= 4
    bits.append(n // 2)
    n %= 2
    bits.append(n)
    bits.reverse()
    return tuple(bits)


def check_tile(maze: NDArray, coords: tuple[int]):
    i, j = coords
    tile = int(maze[i, j])
    msg = f"Maze tile {coords} has incoherent walls"
    walls = get_walls(tile)
    if i == 0:
        if walls[0] == 0:
            raise MazeError(f"{msg} with north border")
    else:
        if walls[0] != get_walls(maze[i - 1][j])[2]:
            raise MazeError(f"{msg} with tile {(i - 1, j)}")
    if i == maze.shape[0] - 1:
        if walls[2] == 0:
            raise MazeError(f"{msg} with south border")
    else:
        if walls[2] != get_walls(maze[i + 1][j])[0]:
            raise MazeError(f"{msg} with tile {(i + 1, j)}")
    if j == 0:
        if walls[3] == 0:
            raise MazeError(f"{msg} with west border")
    else:
        if walls[3] != get_walls(maze[i][j - 1])[1]:
            raise MazeError(f"{msg} with tile {(i, j - 1)}")
    if j == maze.shape[1] - 1:
        if walls[1] == 0:
            raise MazeError(f"{msg} with east border")
    else:
        if walls[1] != get_walls(maze[i][j + 1])[3]:
            raise MazeError(f"{msg} with tile {(i, j + 1)}")

## Maze generator

Initial thought: Wilson algorithm.

* First tile in the maze.
* Random tile makes random walk.
* Looping random walk erases loop.
* Random walk finds maze: becomes part of the maze.
* Repeat until all tiles are part of the maze.

Second algorithm: Aldous-Broder:

* First tile in the maze.
* Point in there makes random walk.
* If unvisited cell: connects both tiles, continue random walk.
* Else: continue random walk.
* Repeat until all tiles are visited.

Complementary algorithm: DFS:

* First tile in the maze.
* Random walk until reaches a tile with no neighbours to visit.
* Bactrack to a tile with neighbours to visit, repeat.
* Repeat until all tiles are visited.

Complementary algorithm: Prim:

* First tile in the maze goes to a set.
* Select a yet to visit neighbour to a tile in the set.
* Connect both tiles.
* If tile has all neighbouring tiles visited, remove it from the set.
* Repeat until set is empty.

To do if I feel like it:

* Hunt & Kill (may be easy after DFS)
* Eller (doable)
* Krunskal (pain)
* Origin Shift (2 painful 4 me)

In [4]:
import random as rng


def manhattan(curr: tuple[int], neigh: tuple[int]) -> int:
    if curr == neigh:
        return 0
    return abs(curr[0] - neigh[0]) + abs(curr[1] - neigh[1])



def connect(maze: NDArray, curr: tuple[int], neigh: tuple[int]) -> None:
    if manhattan(curr, neigh) != 1:
        return
    if curr[0] == neigh[0] + 1:
        maze[curr[0]][curr[1]] -= 1
        maze[neigh[0]][neigh[1]] -= 4
    if curr[0] == neigh[0] - 1:
        maze[curr[0]][curr[1]] -= 4
        maze[neigh[0]][neigh[1]] -= 1
    if curr[1] == neigh[1] - 1:
        maze[curr[0]][curr[1]] -= 2
        maze[neigh[0]][neigh[1]] -= 8
    if curr[1] == neigh[1] + 1:
        maze[curr[0]][curr[1]] -= 8
        maze[neigh[0]][neigh[1]] -= 2


def direction() -> tuple[str, int]:
    if rng.random() < .5:
        axis = "V"
    else:
        axis = "H"
    if rng.random() < .5:
        step = -1
    else:
        step = 1
    return (axis, step)


def count_visited(visited: NDArray) -> int:
    n = 0
    for i in range(visited.shape[0]):
        for j in range(visited.shape[1]):
            if visited[i][j] == 1:
                n += 1
    return n


def wilson_erase(visited: NDArray, path: list[tuple], point: tuple[int]):
    while path[-1] != point:
        visited[path[-1][:]] = 0
        path.pop()


try:
    rng.seed(conf["SEED"])
except KeyError:
    pass

# Initialization
v = -1
while v == -1:
    current = (rng.randrange(0, size[0] - 1), rng.randrange(0, size[1] - 1))
    v = visited[current[:]]
visited[current[0]][current[1]] = 1

# Recursive backtracker
"""lst = [current]
while len(lst) > 0:
    nvis = set()
    h, w = current[:]
    neighs = ((h - 1, w), (h, w + 1), (h + 1, w), (h, w - 1))
    for i in neighs:
        try:
            if True in [i[j] in (-1, maze.shape[j]) for j in [0, 1]]:
                raise IndexError
            if visited[i[:]] == 0:
                nvis.add(i[:])
        except IndexError:
            continue
    if len(nvis) == 0:
        lst.pop()
    else:
        new = rng.choice([*nvis])
        nvis.clear()
        connect(maze, current, new)
        visited[new[:]] = 1
        lst.append(new)
    try:
        current = lst[-1]
    except IndexError:
        pass"""

# Prim Algorithm
"""opened = set()
opened.add(tuple(current))
while len(opened) > 0:
    nvis = set()
    h, w = current[:]
    neighs = ((h - 1, w), (h, w + 1), (h + 1, w), (h, w - 1))
    for i in neighs:
        try:
            if True in [i[j] in (-1, maze.shape[j]) for j in [0, 1]]:
                raise IndexError
            if visited[i[:]] == 0:
                nvis.add(i[:])
        except IndexError:
            continue
    if len(nvis) == 0:
        opened.remove(current)
    else:
        new = rng.choice([*nvis])
        nvis.clear()
        connect(maze, current, new)
        visited[new[:]] = 1
        opened.add(new)
    if len(opened) > 0:
        current = rng.choice([*opened])"""

# Alodus-Broder Algorithm
"""yet = set()
for i in range(maze.shape[0]):
    for j in range(maze.shape[1]):
        if visited[i, j] == 0:
            yet.add((i, j))
while len(yet) > 0:
    direct = direction()
    try:
        if direct[0] == "V":
            neighbour = (current[0] + direct[1], current[1])
            if current[0] + direct[1] in (-1, maze.shape[0]):
                raise IndexError
        else:
            neighbour = (current[0], current[1] + direct[1])
            if current[1] + direct[1] in (-1, maze.shape[1]):
                raise IndexError
    except IndexError:
        continue
    if visited[neighbour[0], neighbour[1]] == -1:
        continue
    elif visited[neighbour[0], neighbour[1]] == 0:
        connect(maze, tuple(current), tuple(neighbour))
        yet.discard(neighbour)
    current = neighbour
    visited[current[0], current[1]] = 1"""

# Wilson Algorithm
yet = set()
for i in range(maze.shape[0]):
    for j in range(maze.shape[1]):
        if visited[i, j] == 0:
            yet.add((i, j))
while len(yet) > 0:
    path = []
    current = rng.choices([*yet], k=1)[0]
    visited[current[0], current[1]] = 2
    path.append(current)
    last_direction = ()
    while current in yet:
        direct = direction()
        while last_direction == direct:
            direct = direction()
        try:
            if direct[0] == "V":
                if current[0] + direct[1] in (-1, maze.shape[0]):
                    raise IndexError
                neighbour = (current[0] + direct[1], current[1])
            else:
                if current[1] + direct[1]  in (-1, maze.shape[1]):
                    raise IndexError
                neighbour = (current[0], current[1] + direct[1])
        except IndexError:
            continue
        if visited[neighbour[0], neighbour[1]] == -1:
            continue
        elif visited[neighbour[0], neighbour[1]] == 1:
            path.append(neighbour)
            break
        elif neighbour in path:
            wilson_erase(visited, path, neighbour)
        current = neighbour
        visited[current[0], current[1]] = 2
        if current not in path:
            path.append(current)
    for tile in path:
        visited[tile[0], tile[1]] = 1
        yet.discard(tile)
    while len(path) > 1:
        connect(maze, path[-2], path[-1])
        current = path[-1]
        path.pop()
    path.clear()

base = "0123456789abcdef"
for i in range(maze.shape[0]):
    for j in range(maze.shape[1]):
        try:
            check_tile(maze, (i, j))
            print(base[maze[i][j]], end="")
        except MazeError as e:
            print(e.msg)
    print()
if 0 in visited:
    print("There are still unvisited tiles")

b915517d17d13d13d113
82ebb879693ae96ad2aa
a87802bc16a8547c56ea
ee92ec056d403b91793e
bbac17c7d17a802ed2c7
86c7e9179452aec39053
c3bd3ac1297eeb906a92
d44546faeafff86c7eae
d51793fc5057fc53b907
d3a92efffefff956c687
b86ac117fbfd52917bc3
aad2d683fafff86ebad2
805413aa9693bc396c3e
ac57eaaec16c414457c3
ed3956e954797c3d553a
978417d4395695417b86
a96b87d3e85569781403
c2d405529691547aad6a
9697817ead6c517ae97a
c547ec55457d5456d456


## Maze solution

A\* algorithm:

```python
# G cost: distance from starting node
# H (heuristic) cost: distance from end node
# h_cost = abs(x_1 - x_2) + abs(y_1 - y_2)
# F cost: G cost + H cost
# open: set of nodes to be evaluated
# closed: set of already evaluated nodes

while True:
    current = "node in open with lowest f_cost"
    remove(current, open)
    add(current, closed)
    
    if current == exit:
        return
    
    for i in neighbours(current):
        if accessible(i) == 0 or i in closed:
            continue

        if f_cost(i) < f_cost(current) or i not in open:
            set_f_cost(i)
            current = parent(i)
            if i not in open:
                add(i, open)
```

In [5]:
from numpy import Inf


class Node():
    def __init__(self, value):
        self.val = value
        self._g = Inf
        self._h = Inf
        self._f = Inf
        self.next = None

    def add_front(self, newnode) -> None:
        if newnode is None:
            return
        newnode.next = self
        self = newnode

    def size(self) -> int:
        first = self
        n = 1
        while self.next != None and self.next is not first:
            n += 1
            self = self.next
        return n

    def get_g(self) -> int:
        return self._g

    def get_h(self) -> int:
        return self._h

    def get_f(self) -> int:
        return self._f

    def _set_f(self, f: int) -> None:
        self._f = f

    def set_g(self, g: int) -> None:
        self._g = g
        self._set_f(self.get_g() + self.get_h())

    def set_h(self, h: int) -> None:
        self._h = h
        self._set_f(self.get_g() + self.get_h())


def neighbours(maze: NDArray, current: tuple[int]) -> list[tuple]:
    n = maze[current[:]]
    neighs = []
    walls = get_walls(n)
    if walls[0] == 0:
        neighs.append((current[0] - 1, current[1]))
    if walls[1] == 0:
        neighs.append((current[0], current[1] + 1))
    if walls[2] == 0:
        neighs.append((current[0] + 1, current[1]))
    if walls[3] == 0:
        neighs.append((current[0], current[1] - 1))
    return neighs


def choose_current(f: NDArray, points: list[Node], exit: tuple[int]) -> Node:
    n = Inf
    f_cost = []
    for i in points:
        p = i.val
        if p == exit:
            return i
        f_cost.append(f[p[:]])
    min_f = min(f_cost)
    to_select = []
    for j in range(len(f_cost)):
        if f_cost[j] == min_f:
            to_select.append(points[j])
    min_h = Inf
    for k in to_select:
        h = manhattan(k.val, exit)
        if h < min_h:
            min_h = h
            current = k
    return current


def check_closed(closed: set[Node], point: tuple[int]) -> bool:
    for i in closed:
        if i.val == point:
            return True
    return False


def check_node(set: set[Node], point = tuple[int]) -> bool:
    for i in set:
        if i.val == point:
            return True
    return False


opened, closed = set(), set()
f_cost = Inf * ones(maze.shape[:], int)
opened.add(Node(entr))
f_cost[entr[:]] = manhattan(entr, exit)
n = 0
while len(opened) > 0:
    n += 1
    current = choose_current(f_cost, list(opened), exit)
    opened.remove(current)
    closed.add(current)
    if current.val == exit:
        break
    for i in neighbours(maze, current.val):
        if check_closed(closed, i) is True:
            continue
        node = Node(i)
        node.next = current
        node.set_g(node.size() - 1)
        node.set_h(manhattan(i, exit))
        if node.get_f() < f_cost[i[:]]:
            f_cost[i[:]] = node.get_f()
            if check_node(opened, i) is True:
                if manhattan(opened[opened.index(i)], entr) > node.get_g():
                    continue
            opened.add(node)

sol = ""
while current.next is not None:
    if current.val[0] == current.next.val[0] + 1:
        sol = "S" + sol
    elif current.val[0] == current.next.val[0] - 1:
        sol = "N" + sol
    elif current.val[1] == current.next.val[1] + 1:
        sol = "E" + sol
    elif current.val[1] == current.next.val[1] - 1:
        sol = "W" + sol
    current = current.next
print(n)
print(sol)

174
SENEEEESSSEEENENESSEESEESESWSWSEESWWSSSESWSESESEEEESSS


## Maze output

In [6]:
txt = ""
with open(conf['OUTPUT_FILE'], 'w') as out:
    print(maze)
    for i in range(size[0]):
        for j in range(size[1]):
            out.write(base[maze[i][j]])
        out.write("\n")

    out.write("\n")
    out.write(f"{entr}          # entry (x,y)\n")
    out.write(f"{exit}          # exit (x,y)\n")
    out.write(sol)

[[11  9  1  5  5  1  7 13  1  7 13  1  3 13  1  3 13  1  1  3]
 [ 8  2 14 11 11  8  7  9  6  9  3 10 14  9  6 10 13  2 10 10]
 [10  8  7  8  0  2 11 12  1  6 10  8  5  4  7 12  5  6 14 10]
 [14 14  9  2 14 12  0  5  6 13  4  0  3 11  9  1  7  9  3 14]
 [11 11 10 12  1  7 12  7 13  1  7 10  8  0  2 14 13  2 12  7]
 [ 8  6 12  7 14  9  1  7  9  4  5  2 10 14 12  3  9  0  5  3]
 [12  3 11 13  3 10 12  1  2  9  7 14 14 11  9  0  6 10  9  2]
 [13  4  4  5  4  6 15 10 14 10 15 15 15  8  6 12  7 14 10 14]
 [13  5  1  7  9  3 15 12  5  0  5  7 15 12  5  3 11  9  0  7]
 [13  3 10  9  2 14 15 15 15 14 15 15 15  9  5  6 12  6  8  7]
 [11  8  6 10 12  1  1  7 15 11 15 13  5  2  9  1  7 11 12  3]
 [10 10 13  2 13  6  8  3 15 10 15 15 15  8  6 14 11 10 13  2]
 [ 8  0  5  4  1  3 10 10  9  6  9  3 11 12  3  9  6 12  3 14]
 [10 12  5  7 14 10 10 14 12  1  6 12  4  1  4  4  5  7 12  3]
 [14 13  3  9  5  6 14  9  5  4  7  9  7 12  3 13  5  5  3 10]
 [ 9  7  8  4  1  7 13  4  3  9  5  6  9  5  4  1  7 11